# 03 · Valid Comparison: Free vs Paid Apps
A naive free-vs-paid comparison is confounded by **genre mix** and **popularity**. This notebook compares them overall, then **within genre** and **within popularity bands**, and tests whether the rating difference is statistically meaningful.

**Rule:** free = `price == 0`, paid = `price > 0`.

In [ ]:
# ── Environment & Spark session (shared by every notebook) ─────────────────────
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/app-store-analytics-spark"
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

# Raw CSV location (override with the APPSTORE_CSV environment variable)
DATA_PATH = os.environ.get("APPSTORE_CSV", os.path.join(PROJECT_DIR, "data", "raw", "apple_appstore_apps.csv"))
PROCESSED_PATH = os.path.join(PROJECT_DIR, "data", "processed", "apps_processed.parquet")
FIG_DIR = os.path.join(PROJECT_DIR, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (SparkSession.builder
         .appName("AppStoreAnalytics")
         .config("spark.sql.adaptive.enabled", "true")                     # Adaptive Query Execution
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")  # merge small shuffle partitions
         .config("spark.sql.adaptive.skewJoin.enabled", "true")            # split skewed partitions
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | master={spark.sparkContext.master} | Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches="tight")

In [ ]:
df = spark.read.parquet(PROCESSED_PATH).cache()
print(f"Loaded {df.count():,} processed apps from {PROCESSED_PATH}")

## 1. Naive (overall) comparison

In [ ]:
overall_pd = (df.groupBy("is_free").agg(
        F.count("*").alias("apps"),
        F.round(F.avg("user_rating"), 3).alias("avg_rating"),
        F.round(F.stddev("user_rating"), 3).alias("sd_rating"),
        F.sum(F.col("is_rated").cast("int")).alias("rated_apps"),
        F.round(F.expr("percentile_approx(rating_count, 0.5)"), 0).alias("median_ratings"),
        F.round(F.avg("size_mb"), 1).alias("avg_size_mb"))
    .toPandas())
overall_pd

### Is the rating difference statistically significant? (Welch t-test + effect size)

In [ ]:
import numpy as np
from math import sqrt
from scipy import stats

g = overall_pd.set_index("is_free")
m1, s1, n1 = g.loc[True, "avg_rating"], g.loc[True, "sd_rating"], g.loc[True, "rated_apps"]
m0, s0, n0 = g.loc[False, "avg_rating"], g.loc[False, "sd_rating"], g.loc[False, "rated_apps"]
t, p = stats.ttest_ind_from_stats(m1, s1, n1, m0, s0, n0, equal_var=False)
pooled = sqrt(((n1 - 1) * s1**2 + (n0 - 1) * s0**2) / (n1 + n0 - 2))
print(f"Free mean={m1:.3f} (n={n1:,}) | Paid mean={m0:.3f} (n={n0:,})")
print(f"Welch t={t:.2f}, p={p:.3g}, Cohen's d={(m1 - m0) / pooled:.3f}")
print("With very large n even tiny differences are 'significant' - judge practical relevance by effect size.")

## 2. Stratified comparison (controls for genre and popularity)

In [ ]:
by_genre = (df.groupBy("primary_genre", "is_free").agg(F.count("*").alias("apps"), F.avg("user_rating").alias("avg_rating"))
              .groupBy("primary_genre")
              .pivot("is_free", [True, False]).agg(F.first("avg_rating"))
              .withColumnRenamed("true", "free_rating").withColumnRenamed("false", "paid_rating")
              .withColumn("paid_minus_free", F.round(F.col("paid_rating") - F.col("free_rating"), 3))
              .dropna().orderBy(F.desc("paid_minus_free")))
genre_gap = by_genre.toPandas()
print(f"Paid apps rated higher in {(genre_gap.paid_minus_free > 0).sum()} of {len(genre_gap)} genres")
genre_gap.head(10)

In [ ]:
pop_q = df.filter("popularity_score > 0").approxQuantile("popularity_score", [0.5, 0.9], 0.001)
band_pd = (df.withColumn("popularity_band",
                F.when(F.col("popularity_score") >= pop_q[1], "Top 10%")
                 .when(F.col("popularity_score") >= pop_q[0], "Above median")
                 .otherwise("Below median"))
           .groupBy("popularity_band", "is_free").agg(F.avg("user_rating").alias("avg_rating"), F.count("*").alias("apps"))
           .toPandas())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=genre_gap.sort_values("paid_minus_free"), x="paid_minus_free", y="primary_genre", ax=axes[0])
axes[0].axvline(0, color="black", lw=1); axes[0].set_title("Rating gap (paid - free) within each genre"); axes[0].set_ylabel("")
sns.barplot(data=band_pd, x="popularity_band", y="avg_rating", hue="is_free", ax=axes[1],
            order=["Below median", "Above median", "Top 10%"])
axes[1].set_title("Average rating by popularity band: free vs paid")
plt.tight_layout(); save(fig, "03_free_vs_paid_stratified.png"); plt.show()

## 3. Validity & caveats
- **Confounding:** genre and popularity are controlled by stratification; developer reputation and app age are not observed.
- **Hidden monetisation:** free apps often earn through in-app purchases and ads, which `price` does not capture.
- **Survivorship / selection bias:** only apps still listed are observed; unrated apps are excluded from rating averages.
- **Next step:** propensity-score matching on genre, size and developer to compare like-for-like apps.